# 03 — Proposed ResNet-18 with staged fine-tuning

The proposed model is an ImageNet-pretrained ResNet-18 with dropout, mild X-ray-safe augmentation, weighted cross-entropy, label smoothing, and three fine-tuning stages: head, layer4, then full network with discriminative layer learning rates.

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

In [ ]:
from src.utils import load_config
cfg = load_config("configs/proposed.yaml")
for stage in cfg["training"]["stages"]:
    print(stage)

Training commands (about 21–31 minutes per seed on the CPU host):

```powershell
python -m src.train --config configs/proposed.yaml --seed 42
python -m src.evaluate --checkpoint checkpoints/proposed_seed42.pt --splits val test
```

All reported seeds use batch size 16.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

hist = pd.read_csv("results/proposed/seed42/history.csv")
hist[["epoch", "stage", "train_loss", "val_loss", "val_macro_f1", "val_macro_auroc"]].tail(12)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(hist.epoch, hist.train_loss, label="train")
axes[0].plot(hist.epoch, hist.val_loss, label="validation")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Cross-entropy"); axes[0].legend()
axes[1].plot(hist.epoch, hist.val_macro_f1, color="darkgreen")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Validation macro-F1")
fig.suptitle("Proposed ResNet-18, seed 42")
fig.tight_layout()
plt.show()

In [ ]:
import json
for seed in (42, 43, 44):
    m = json.load(open(f"results/proposed/seed{seed}/metrics_test.json"))
    print(seed, {k: round(m[k], 4) for k in ("accuracy", "balanced_accuracy", "macro_f1", "macro_auroc", "ece")})

Test macro-F1: 0.8273, 0.8343, 0.8426 for seeds 42, 43, 44. Mean ± std: 0.8347 ± 0.0077. The paired bootstrap difference versus the baseline is positive for all three seeds.